# 01 · Data Exploration

Understand the dataset before modelling: schema, class balance, missing values, and feature distributions (supports **RQ1**).

- **Inputs:** `data/sample/sample_prs.csv`
- **Outputs:** Inline tables and charts; insights to guide preprocessing (nb 02).

> ⚠️ **Sample vs. real data.** This notebook runs on the committed 10-row synthetic sample so the toolchain works without PRismBench. The sample has singleton classes, so metrics here are *illustrative only*. Each `TODO` marks where the real dataset in `data/raw/` plugs in.

In [ ]:
# --- Standard setup: locate project root, add src/ to path, load helpers ---
import sys
from pathlib import Path

import pandas as pd


def find_project_root(start: Path) -> Path:
    """Walk upwards until we find the repo root (has pyproject.toml + src/pr_risk)."""
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists() and (p / "src" / "pr_risk").exists():
            return p
    return start


PROJECT_ROOT = find_project_root(Path.cwd())
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

pd.set_option("display.max_columns", 50)
SAMPLE_CSV = PROJECT_ROOT / "data" / "sample" / "sample_prs.csv"
print("Project root :", PROJECT_ROOT)
print("Sample CSV   :", SAMPLE_CSV.name, "| exists:", SAMPLE_CSV.exists())

In [ ]:
import matplotlib.pyplot as plt

try:
    import seaborn as sns

    sns.set_theme(style="whitegrid")
    HAS_SNS = True
except ImportError:  # seaborn is optional; matplotlib is enough
    HAS_SNS = False
print("seaborn available:", HAS_SNS)

## 1. Load the dataset
We load via the package helper so notebooks stay thin.

In [ ]:
from pr_risk.data.load_data import load_csv

df = load_csv(SAMPLE_CSV)
print("shape:", df.shape)
df.head()

## 2. Schema & data types

In [ ]:
schema = df.dtypes.rename("dtype").to_frame()
schema["non_null"] = df.notna().sum()
schema["n_unique"] = df.nunique()
schema

## 3. Class balance
Both targets matter: `is_risky` (binary/ternary) and `risk_type` (multi-class).

In [ ]:
is_risky_counts = df["is_risky"].value_counts().sort_index()
risk_type_counts = df["risk_type"].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
is_risky_counts.plot(kind="bar", ax=axes[0], color="#4C72B0")
axes[0].set_title("is_risky (0=non-risky, 1=risky, 2=unsure)")
axes[0].set_xlabel("is_risky")
axes[0].set_ylabel("count")
risk_type_counts.plot(kind="barh", ax=axes[1], color="#55A868")
axes[1].set_title("risk_type distribution")
axes[1].set_xlabel("count")
plt.tight_layout()
plt.show()

display(is_risky_counts.to_frame("count"))
display(risk_type_counts.to_frame("count"))

## 4. Missing values
Real PRismBench data will have far more missingness than the sample.

In [ ]:
missing = df.isna().sum().rename("missing").to_frame()
missing["pct"] = (missing["missing"] / len(df) * 100).round(1)
missing.sort_values("missing", ascending=False)

## 5. Numeric feature distributions

In [ ]:
numeric_cols = [
    "files_changed", "lines_added", "lines_deleted",
    "commits_count", "comments_count", "reviewers_count",
]
display(df[numeric_cols].describe().T)

axes = df[numeric_cols].hist(figsize=(12, 7), bins=8, color="#4C72B0")
plt.suptitle("Numeric metadata distributions (sample)")
plt.tight_layout()
plt.show()

## 6. Risk signal vs. metadata
Do risky PRs look different on average? (Trends here are not significant on 10 rows.)

In [ ]:
by_risk = df.groupby("is_risky")[numeric_cols + ["ci_failed"]].mean(numeric_only=True)
display(by_risk)

# CI failure rate by risk label
ci_rate = df.assign(ci_failed=df["ci_failed"].astype(bool)).groupby("is_risky")["ci_failed"].mean()
ci_rate.plot(kind="bar", color="#C44E52", figsize=(5, 3), title="CI failure rate by is_risky")
plt.ylabel("share CI failed")
plt.tight_layout()
plt.show()

## Next steps / TODO (real data)
- Point `SAMPLE_CSV` at the real file in `data/raw/` and re-run.
- Add PRismBench-specific columns (text length, code-churn, author history, review timeline).
- Inspect duplicates, outliers, and label noise; quantify class imbalance for `risk_type`.
- Feed findings into **02 · Preprocessing**.